In [1]:
import os
import json
import pandas as pd
import seaborn as sns
import yaml

In [2]:
wandb_dir = "/scratch/nia4240/compute-better_spent-scratch/wandb"

In [3]:
rows = []
for folder in os.listdir(wandb_dir):
    if folder.startswith("run-") and os.path.isdir(os.path.join(wandb_dir, folder)):
        summary_path = os.path.join(wandb_dir, folder, 'files/wandb-summary.json')
        config_path = os.path.join(wandb_dir, folder, 'files/config.yaml')
        
        if os.path.exists(summary_path):
            with open(summary_path, 'r') as f:
                summary_data = json.load(f)

        if os.path.exists(config_path):
            with open(config_path, 'r') as f:
                config_data = yaml.safe_load(f)
        
        config_clean = {k: v['value'] for k, v in config_data.items() if k != 'wandb_version'}
        rows.append({**config_clean, **summary_data})

df = pd.DataFrame(rows)

In [4]:
# drop rows for which the column 'run_name' is not 'muP_test' or 'muP_test_noQKnorm'
df['run_name_prefix'] = df['run_name'].str.split('/').str[0]
df = df[df['run_name_prefix'].isin(['muP_test', 'muP_test_noQKnorm'])]
df['condition'] = df.apply(lambda x: (x['run_name_prefix'], x['alt_attn_config'][:15]), axis=1)

In [ ]:
sns.lineplot(x='lr', y='test_loss', hue='scale_factor', style='condition', data=df, marker='o')